## Overview: Human Annotation Data Processing

We process the raw human survey data (Excel format) containing:

- 28 human annotators  
- 180 questions  
- Some questions intentionally repeated (for test–retest reliability analysis)

Our goal is to separate:

1. **Repeated questions** → for internal consistency (test–retest) evaluation  
2. **Non-repeated questions** → for population-wise disagreement evaluation  

This ensures that internal reliability and cross-annotator disagreement are evaluated independently and cleanly.

## Final Dataset Structure

After preprocessing, we obtain:

| File | Purpose |
|------|---------|
| `human_dup.csv` | Internal (test–retest) evaluation |
| `dup_questions.csv` | List of duplicated questions |
| `human_all.csv` | Full tidy dataset |
| `human_none_dup.csv` | Clean dataset for population evaluation |
| `none_dup_questions.csv` | Final unique non-duplicated question set |

---

This separation allows us to clearly distinguish:

- **Within-annotator stability**  
- **Across-annotator disagreement**  

which are conceptually and statistically different levels of analysis.

In [1]:
from pathlib import Path
from mhdr.dataloader.io import read_excel, read_csv, save_csv
from mhdr.dataloader.process_human_data import extract_human_dup_with_qid, extract_dup_questions, remove_dup_questions, extract_human_all_with_qid   
INPUT_DIR = Path.cwd() / "input"
OUTPUT_DIR = Path.cwd() / "output"
TEMP_DIR = Path.cwd() / "temp"
LLM_DATA = read_csv(INPUT_DIR / "questions_master.csv")
HUMAN_DATA = read_excel(INPUT_DIR / "HUMAN.xlsx")


## Data Separation Strategy

### 1. Extract Duplicated Questions (Internal Evaluation)

We identify questions that appear more than once (based on normalized text) and extract:

- All responses to duplicated questions  
- Unique duplicated question list  

Generated files:

- `human_dup.csv`  
  → All duplicated-question responses (tidy format: qid, text, answer, source)

- `dup_questions.csv`  
  → Unique list of duplicated questions

This dataset is used for:

- Test–retest reliability  
- Internal human consistency analysis  

---

### 2. Extract All Human Responses

We convert the full wide survey table into tidy format:

- One row per (human, question)
- With assigned `qid`

Generated file:

- `human_all.csv`

This contains all 180 questions and all responses (including duplicates).

In [2]:
# 1) duplicated rows
dup = extract_human_dup_with_qid(HUMAN_DATA, [LLM_DATA])
save_csv(dup, OUTPUT_DIR / "human_dup.csv")

dup_questions = extract_dup_questions(dup)
save_csv(dup_questions, OUTPUT_DIR / "dup_questions.csv")

print("dup:", dup.shape, "dup_questions:", dup_questions.shape)
print("dup columns:", dup.columns.tolist())
print("dup_questions columns:", dup_questions.columns.tolist())


all_rows = extract_human_all_with_qid(HUMAN_DATA, [LLM_DATA])  
save_csv(all_rows, OUTPUT_DIR / "human_all.csv")
print("all_rows:", all_rows.shape)
print("all_rows columns:", all_rows.columns.tolist())



dup: (1456, 5) dup_questions: (52, 2)
dup columns: ['qid', 'text', 'answer', 'source', 'text_norm']
dup_questions columns: ['qid', 'text']
all_rows: (5039, 5)
all_rows columns: ['qid', 'text', 'answer', 'source', 'text_norm']


### 3. Remove Duplicated Questions (Population Evaluation)

We remove all duplicated-question rows from the full dataset to obtain a clean, non-repeated question set.

Generated files:

- `human_none_dup.csv`  
  → All responses to non-duplicated questions

- `none_dup_questions.csv`  
  → Unique list of non-duplicated questions

This dataset is used for:

- Population-wise entropy analysis  
- Human–LLM disagreement comparison  
- Structural stability evaluation  

---

In [3]:
# 3) remove duplicated qids from ALL rows
none_dup = remove_dup_questions(all_rows, dup_questions) 
save_csv(none_dup, OUTPUT_DIR / "human_none_dup.csv")
print("none_dup:", none_dup.shape)
print("none_dup columns:", none_dup.columns.tolist())

none_dup_questions = extract_dup_questions(none_dup)
save_csv(none_dup_questions, OUTPUT_DIR / "none_dup_questions.csv")
print("none_dup_questions:", none_dup_questions.shape)
print("none_dup_questions columns:", none_dup_questions.columns.tolist())

none_dup: (3583, 5)
none_dup columns: ['qid', 'text', 'answer', 'source', 'text_norm']
none_dup_questions: (78, 2)
none_dup_questions columns: ['qid', 'text']
